# ViPRA 2026 - chuẩn bị submission cho Viper

Chạy tại thư mục mã nguồn. Đọc SUBMISSION_GUIDE_VI.md trước. Chưa có adapter thì dừng ở phần khai báo, các bước còn lại chạy sau khi có checkpoint. Notebook không tự train hoặc thuê GPU. BTC chấm qua API online trong ca một giờ.


In [ ]:
from pathlib import Path
import subprocess, sys
RUN_DIR = None  # Thư mục experiment/sweep hoàn tất, có adapter và run.json
TRAIN_FILE = None  # train_dataset_reviewed.csv đã dùng trong run
DEV_FILE = None
BASE_URL = None  # https://tên-miền-thật đã trỏ về máy GPU
OFFICIAL_NOTATION = None  # CSV chính thức BTC cung cấp
PUBLIC_TEST = None  # File CSV/JSON/JSONL khi nhận được
DEPLOYMENT = Path("deployment")
def run(*args):
    return subprocess.run([sys.executable, "-m", *map(str, args)], check=True)


## 1. Chuẩn bị adapter sau khi train
Dùng môi trường có safetensors và PyTorch (môi trường training hiện tại). Chép cả deployment/ sang máy hosting.


In [ ]:
if RUN_DIR is None:
    print("Để bước này sau: chưa có checkpoint LoRA.")
else:
    run("submission.prepare", "--run-dir", RUN_DIR, "--team-name", "Viper", "--output-dir", DEPLOYMENT)


## 2. Chạy máy GPU
Trên Linux NVIDIA GPU, làm theo phần Docker trong SUBMISSION_GUIDE_VI.md: điền deploy/.env, đặt deployment/ đúng chỗ, chạy Docker Compose. Hoặc tạo môi trường .venv-serving riêng, cài requirements-vllm.txt rồi chạy submission.launch trong tmux. Không cài vLLM đè lên môi trường training và không dùng notebook đóng ngay sau khi bật API.

URL công khai phải truy cập được từ máy khác. Cổng 80/443 mở, DNS chính xác. Giữ máy online toàn bộ ca chấm.


## 3. Kiểm tra URL bằng câu thử trước ca chấm
Cài requirements-serving.txt trên máy gọi API. Câu thử dùng ba ví dụ có sẵn; không gọi public test để warm-up.


In [ ]:
if BASE_URL is None:
    print("Chưa có URL API công khai.")
else:
    run("submission.check", "--base-url", BASE_URL, "--output", "outputs/preflight.json")


## 4. Sinh file prediction khi có public test
Chỉ đọc ID và câu hỏi. Nếu BTC chấm bằng API, BTC tự gọi API trong ca chấm; không cần file này để thay thế API.


In [ ]:
if PUBLIC_TEST is None or BASE_URL is None:
    print("Chờ file public test và URL.")
else:
    run("submission.predict", "--input", PUBLIC_TEST, "--endpoint", BASE_URL.rstrip("/") + "/predict", "--output-dir", "outputs/public_test_run1")


## 5. Chuẩn bị notation chính thức
Mặc định giữ các quy ước canonical của BTC; chỉ khai báo khác biệt đã quyết định. Không thay toàn bộ mu thành u vì còn biểu diễn độ từ thẩm.


In [ ]:
if OFFICIAL_NOTATION is None:
    print("Chờ bảng notation chính thức từ BTC.")
else:
    run("submission.notation", "--official", OFFICIAL_NOTATION, "--output", "outputs/notation_mapping.csv")


## 6. PDF một trang và Viper.zip
Cài requirements-package.txt ở máy đóng gói. Dataset phải đúng hash của run. Xem PDF trước khi nộp. Nếu thiếu font, thêm --font với file TTF hỗ trợ tiếng Việt.


In [ ]:
if TRAIN_FILE is None or DEV_FILE is None or not DEPLOYMENT.is_dir():
    print("Chưa đủ run/dataset để tạo PDF cuối cùng.")
else:
    run("submission.report", "--train-file", TRAIN_FILE, "--dev-file", DEV_FILE, "--deployment", DEPLOYMENT, "--output", "outputs/solution.pdf")


In [ ]:
if BASE_URL is None or OFFICIAL_NOTATION is None or not Path("outputs/solution.pdf").is_file():
    print("Chưa đủ hồ sơ để đóng gói.")
else:
    run("submission.package", "--deployment", DEPLOYMENT, "--solution", "outputs/solution.pdf", "--notation", "outputs/notation_mapping.csv", "--official-notation", OFFICIAL_NOTATION, "--base-url", BASE_URL, "--preflight", "outputs/preflight.json")


## 7. Thao tác với BTC
Đăng ký ca chấm một giờ qua form BTC thông báo; tải outputs/submission_final/Viper.zip lên form nộp. Kiểm tra deadline mới trong Zalo/website vì PDF còn ghi TBA. Bật API và warm-up trước ca, giữ online đủ giờ; lưu deployment/logs/ riêng.
